# Evaluation run

Serves the model with vLLM on the GPU and runs the warehouse, dbt, the MCP server and the agents alongside it.
Needs an A100 (80 GB) with high RAM. Runs are traced in Langfuse when the `LANGFUSE_*` secrets are set.

Results are written to Drive after each incident; incidents that already have a result are skipped.

In [1]:
import os, subprocess

repo = "/content/responder"
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/arihaaant/data-incident-responder.git", repo], check=True)
print("code at", subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/responder
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

code at da14f8d incident report shows column profiles

/content/responder
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 53.1 MB/s eta 0:00

In [2]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

tracing on


## Model server

In [3]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

model up after 7.8 min


## Warehouse and tests

In [4]:
import subprocess, sys
!python -m responder.warehouse --path kit/source.duckdb
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout[-1500:])
assert tests.returncode == 0, "test suite failed"

built kit/source.duckdb
.......................................................                  [100%]
55 passed in 32.81s



## Responder and baselines on every incident, three runs each

In [6]:
from google.colab import drive

drive.mount("/content/drive")
run_name = "final"
out = f"/content/drive/MyDrive/data-incident-responder/{run_name}"
os.makedirs(out, exist_ok=True)

Mounted at /content/drive


In [7]:
print("=== rules ===")
!python -m responder.run --system rules --workers 6 --out {out}/rules
for r in range(1, 4):
    for system in ["multi", "single"]:
        print(f"\n=== {system}, run {r} ===")
        !python -m responder.run --system {system} --workers 8 --out {out}/{system}-{r}


=== multi ===
[10/11/26 19:19:52] INFO     Processing request of type            ]8;id=636943;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=583787;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/11/26 19:19:52] INFO     Processing request of type            ]8;id=73109;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=240576;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/11/26 19:19:52] INFO     Processing request of type            ]8;id=336832;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=843395;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/s

## Results

In [8]:
for name in ["rules"] + [f"{s}-{r}" for r in range(1, 4) for s in ["multi", "single"]]:
    print(f"\n=== {name} ===")
    !python -m responder.report {out}/{name}


=== multi ===
                   n    root   cause  action  resolved  tokens
overall           50   90.0%   96.0%   92.0%     92.0%   18626
duplicates        12   83.3%  100.0%   83.3%     83.3%   23964
late_data         12   91.7%   91.7%   91.7%     91.7%   20328
null_spike        12   91.7%  100.0%  100.0%    100.0%   17725
schema_drift      14   92.9%   92.9%   92.9%     92.9%   13365
dev               50   90.0%   96.0%   92.0%     92.0%   18626

resolved 46/50 (95% CI 81 to 97%), fixes passing the hidden check 30/34

=== single ===
                   n    root   cause  action  resolved  tokens
overall           50   76.0%   80.0%   94.0%     94.0%   23859
duplicates        12   91.7%  100.0%   91.7%     91.7%   27491
late_data         12   83.3%   91.7%   91.7%     91.7%   29808
null_spike        12   91.7%   91.7%  100.0%    100.0%   19120
schema_drift      14   42.9%   42.9%   92.9%     92.9%   19709
dev               50   76.0%   80.0%   94.0%     94.0%   23859

resolved 47/5